# EfficientNet-B0 Implementation

This notebook initializes ImageNet-pretrained EfficientNet-B0 with the existing Sorghum cultivar mapping and verifies one forward, loss, and backward pass.

In [1]:
# 1. Imports
import sys
from pathlib import Path

import pandas as pd
import torch
from torch import nn
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset import SorghumDataset, load_class_to_idx, make_loader
from src.preprocessing import IMAGE_SIZE, get_train_transform

In [2]:
# 2. Configuration and paths
RESULTS_DIR = PROJECT_ROOT / 'results'
TRAIN_MANIFEST = RESULTS_DIR / 'train.csv'
CLASS_MAPPING = RESULTS_DIR / 'class_to_index.json'
IMAGE_DIR = PROJECT_ROOT.parent / 'Dataset'
BATCH_SIZE = 32
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Project root: {PROJECT_ROOT}')
print(f'Device: {DEVICE}')

Project root: C:\Users\nihar\OneDrive\Desktop\Water To Pixels\pixels-to-water
Device: cuda


In [3]:
# 3. Load the existing class mapping
class_to_idx = load_class_to_idx(CLASS_MAPPING)
print(f'Loaded class mapping from {CLASS_MAPPING}.')

Loaded class mapping from C:\Users\nihar\OneDrive\Desktop\Water To Pixels\pixels-to-water\results\class_to_index.json.


In [4]:
# 4. Determine the number of cultivar classes
num_classes = len(class_to_idx)
print(f'Number of classes: {num_classes}')

Number of classes: 100


In [5]:
# 5. Load pretrained EfficientNet-B0
weights = EfficientNet_B0_Weights.IMAGENET1K_V1
model = efficientnet_b0(weights=weights)
model = model.to(DEVICE)
print(f'Loaded pretrained weights: {weights}')

Loaded pretrained weights: EfficientNet_B0_Weights.IMAGENET1K_V1


In [6]:
# 6. Replace the final classifier
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
model = model.to(DEVICE)
print(f'Classifier output features: {model.classifier[1].out_features}')

Classifier output features: 100


In [7]:
# 7. Display the model architecture
print(model)

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [8]:
# 8. Display model parameter counts
total_parameters = sum(parameter.numel() for parameter in model.parameters())
trainable_parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
print(f'Number of classes: {num_classes}')
print(f'Total parameters: {total_parameters:,}')
print(f'Trainable parameters: {trainable_parameters:,}')

Number of classes: 100
Total parameters: 4,135,648
Trainable parameters: 4,135,648


In [9]:
# 9. Load one batch from the existing training dataset
train_manifest = pd.read_csv(TRAIN_MANIFEST)
if not set(train_manifest['cultivar'].unique()).issubset(class_to_idx):
    raise ValueError('Training manifest contains cultivars absent from the existing class mapping.')
train_dataset = SorghumDataset(train_manifest, IMAGE_DIR, get_train_transform(), class_to_idx)
train_loader = make_loader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)
images, labels = next(iter(train_loader))
print(f'Input tensor shape: {tuple(images.shape)}')
assert tuple(images.shape[1:]) == (3, IMAGE_SIZE, IMAGE_SIZE), f'Unexpected input shape: {tuple(images.shape)}'
assert labels.ndim == 1 and torch.all((labels >= 0) & (labels < num_classes)), 'Invalid training labels.'

Input tensor shape: (32, 3, 224, 224)


In [10]:
# 10-13. Forward pass, output shape, loss, and backward verification
model.train()
images = images.to(DEVICE)
labels = labels.to(DEVICE)
outputs = model(images)
print(f'Output tensor shape: {tuple(outputs.shape)}')
expected_output_shape = (images.shape[0], num_classes)
assert tuple(outputs.shape) == expected_output_shape, f'Expected {expected_output_shape}, got {tuple(outputs.shape)}'
criterion = nn.CrossEntropyLoss()
loss = criterion(outputs, labels)
assert torch.isfinite(loss), 'Loss is not finite.'
print('CrossEntropyLoss calculation works.')
loss.backward()
assert any(parameter.grad is not None for parameter in model.parameters() if parameter.requires_grad), 'No gradients were produced.'
print('Backward pass works.')
print('EfficientNet-B0 implementation verified successfully.')

Output tensor shape: (32, 100)


CrossEntropyLoss calculation works.


Backward pass works.
EfficientNet-B0 implementation verified successfully.
